# Lab 09: Cost-Quality-Latency Pareto Analysis

## Objectives
- Define model candidates with realistic pricing
- Evaluate quality across models
- Measure cost and latency characteristics
- Compute Pareto frontier
- Create interactive visualizations
- Build decision framework for model selection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Set

np.random.seed(42)
print('Imports successful!')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

## Step 1: Define Model Candidates
Define 7 LLM models with realistic pricing information.

In [ ]:
models_list = [
    {'name': 'GPT-4o', 'provider': 'OpenAI', 'input': 0.005, 'output': 0.015, 'latency': 450, 'quality': 0.92},
    {'name': 'Claude Sonnet', 'provider': 'Anthropic', 'input': 0.003, 'output': 0.015, 'latency': 480, 'quality': 0.90},
    {'name': 'Claude Opus', 'provider': 'Anthropic', 'input': 0.015, 'output': 0.075, 'latency': 520, 'quality': 0.95},
    {'name': 'Gemini Flash', 'provider': 'Google', 'input': 0.000075, 'output': 0.0003, 'latency': 200, 'quality': 0.78},
    {'name': 'Qwen3.5-9B', 'provider': 'Alibaba', 'input': 0.0005, 'output': 0.0005, 'latency': 350, 'quality': 0.72},
    {'name': 'Llama Scout', 'provider': 'Meta', 'input': 0.0001, 'output': 0.0004, 'latency': 300, 'quality': 0.68},
    {'name': 'GPT-4o-mini', 'provider': 'OpenAI', 'input': 0.00015, 'output': 0.0006, 'latency': 350, 'quality': 0.82}
]
models_df = pd.DataFrame(models_list)
print('Model Candidates:')
print(models_df[['name', 'provider', 'quality', 'latency']].to_string(index=False))
print(f'\\nTotal models: {len(models_list)}')

## Step 2: Quality Evaluation
Realistic quality scores from benchmark performance.

In [ ]:
pareto_data = []
for model in models_list:
    quality = model['quality']
    cost = ((model['input'] * 500 + model['output'] * 500) / 1000) * 0.001
    latency = model['latency']
    pareto_data.append((quality, cost, latency))

print('Model Performance Metrics:')
for m, (q, c, l) in zip(models_list, pareto_data):
    print(f"{m['name']:<20} Quality: {q:.2f}  Cost: ${c:.6f}  Latency: {l}ms")

## Step 3: Cost Analysis
Estimate monthly costs at different usage levels.

In [ ]:
print('\\nMonthly Cost (Medium Usage: 1000 calls/day, 500 in/500 out tokens)')
for m in models_list:
    daily_cost = 1000 * ((m['input'] * 500 + m['output'] * 500) / 1000)
    monthly = daily_cost * 30
    print(f"{m['name']:<20} ${monthly:>8.2f}/month")

## Step 4: Pareto Frontier
Identify non-dominated models using multi-objective optimization.

In [ ]:
def pareto_frontier(data: List[Tuple], maximize: List[bool]) -> Set[int]:
    frontier = set()
    n = len(data)
    for i in range(n):
        dominated = False
        for j in range(n):
            if i == j: continue
            dominates = True
            for d in range(len(data[0])):
                condition = (data[j][d] >= data[i][d]) if maximize[d] else (data[j][d] <= data[i][d])
                if not condition: dominates = False; break
            if dominates: dominated = True; break
        if not dominated: frontier.add(i)
    return frontier

frontier_idx = pareto_frontier(pareto_data, [True, False, False])
print(f'Pareto Frontier Size: {len(frontier_idx)} models')
print('\\nFrontier Members:')
for idx in sorted(frontier_idx):
    m = models_list[idx]
    print(f"  {m['name']:<20} Quality: {m['quality']:.2f}  Cost: ${pareto_data[idx][1]:.6f}")

## Step 5: Visualization
2D scatter plots showing trade-offs between dimensions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
for idx, m in enumerate(models_list):
    q, c, l = pareto_data[idx]
    marker, color = ('*', 'red') if idx in frontier_idx else ('o', 'blue')
    size = 300 if idx in frontier_idx else 100
    ax.scatter(c, q, s=size, marker=marker, color=color, alpha=0.7)
    ax.annotate(m['name'], (c, q), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Cost per Call (USD, log scale)')
ax.set_ylabel('Quality Score')
ax.set_title('Quality vs Cost')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax = axes[1]
for idx, m in enumerate(models_list):
    q, c, l = pareto_data[idx]
    marker, color = ('*', 'red') if idx in frontier_idx else ('o', 'blue')
    size = 300 if idx in frontier_idx else 100
    ax.scatter(l, q, s=size, marker=marker, color=color, alpha=0.7)
    ax.annotate(m['name'], (l, q), fontsize=8, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('Latency (ms)')
ax.set_ylabel('Quality Score')
ax.set_title('Quality vs Latency')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pareto_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pareto visualization saved.')

## Step 6: Decision Framework

### Use Case Recommendations

**Cost-Sensitive**: Llama Scout or Gemini Flash
- Lowest cost per token
- Acceptable quality for simple tasks
- Monthly cost: $3-8

**Quality-First**: Claude Opus or GPT-4o
- Best performance across benchmarks
- Handles complex reasoning
- Worth premium for high-stakes decisions

**Balanced**: Claude Sonnet or GPT-4o-mini
- Good quality-to-cost ratio
- Reliable performance
- Sweet spot for most applications

**Latency-Sensitive**: Gemini Flash or Qwen3.5
- Fastest response times (200-350ms)
- Acceptable quality
- Real-time applications

**Mission-Critical**: Claude Opus
- Highest quality
- Best reasoning ability
- Final verification step

### Key Insights
1. Pareto frontier contains only models worth considering
2. 1000x cost difference across models
3. Quality ranges from 68% to 95%
4. Claude Sonnet offers best value for most use cases
5. Consider multi-model strategies (tiering, ensembles)